### Self Discovery Lab: The Financial RAG Challenge

Data Source: Databricks OfficeQA

Chosen Data Format: Transformed .txt / Markdown files

Years Used: 2022, 2023, 2024, 2025

Vector Database: ChromaDB
  
Embedding Model: sentence-transformers/all-MiniLM-L6-v2  

This notebook compares a Baseline RAG system and an Engineered RAG system using U.S. Treasury Bulletin documents.

Note: Because I did not use an OpenAI API key, I evaluated the generator manually by checking whether the retrieved context contained enough evidence to support the gold answer.

In [ ]:
!pip install -q datasets huggingface_hub chromadb sentence-transformers langchain-text-splitters pandas tqdm openai

In [ ]:
import chromadb
import datasets
import sentence_transformers
import pandas as pd

print("All imports worked.")
print("ChromaDB version:", chromadb.__version__)

In [ ]:
!git clone https://github.com/databricks/officeqa.git

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "databricks/officeqa",
    data_files="officeqa_full.csv",
    split="train"
)

qa = dataset.to_pandas()

print("Total questions:", len(qa))
print(qa.columns)
qa.head()

In [ ]:
import re
from pathlib import Path

SELECTED_YEARS = [2022, 2023, 2024, 2025]
YEAR_SET = set(SELECTED_YEARS)

def parse_source_files(source_text):
    """
    Extract Treasury Bulletin file names from the source_files column.
    This handles cases where multiple files are separated by new lines.
    """
    if source_text is None:
        return []

    text = str(source_text)

    matches = re.findall(
        r"treasury_bulletin_(\d{4})_(\d{2})\.txt",
        text
    )

    files = [
        f"treasury_bulletin_{year}_{month}.txt"
        for year, month in matches
    ]

    return sorted(set(files))

def get_year(filename):
    match = re.search(r"treasury_bulletin_(\d{4})_(\d{2})\.txt", filename)
    return int(match.group(1)) if match else None

def get_month(filename):
    match = re.search(r"treasury_bulletin_(\d{4})_(\d{2})\.txt", filename)
    return int(match.group(2)) if match else None

qa["source_files_parsed"] = qa["source_files"].apply(parse_source_files)
qa["source_years"] = qa["source_files_parsed"].apply(
    lambda files: sorted(set(get_year(f) for f in files))
)

print("Example parsed files:")
print(qa.loc[0, "source_files"])
print("Parsed as:", qa.loc[0, "source_files_parsed"])

print("\nExample with multiple files if available:")
multi = qa[qa["source_files_parsed"].apply(lambda x: len(x) > 1)]
if len(multi) > 0:
    print(multi.iloc[0]["source_files"])
    print("Parsed as:", multi.iloc[0]["source_files_parsed"])
else:
    print("No multi-file examples found.")

In [ ]:
year_counts = (
    qa.explode("source_years")
      .groupby("source_years")
      .size()
      .sort_index()
)

print("Answer-key question counts by year:")
print(year_counts.tail(30))

In [ ]:
qa_filtered = qa[
    qa["source_files_parsed"].apply(
        lambda files: len(files) > 0 and all(get_year(f) in YEAR_SET for f in files)
    )
].reset_index(drop=True)

print("Selected years:", SELECTED_YEARS)
print("Number of filtered questions:", len(qa_filtered))

qa_filtered[["uid", "question", "answer", "source_files_parsed", "difficulty"]].head()

In [ ]:
year_counts = (
    qa.explode("source_years")
      .groupby("source_years")
      .size()
      .sort_index()
)

print("Recent answer-key question counts:")
print(year_counts.tail(20))

In [ ]:
SELECTED_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
YEAR_SET = set(SELECTED_YEARS)

qa_filtered = qa[
    qa["source_files_parsed"].apply(
        lambda files: len(files) > 0 and all(get_year(f) in YEAR_SET for f in files)
    )
].reset_index(drop=True)

print("Selected years:", SELECTED_YEARS)
print("Number of filtered questions:", len(qa_filtered))

qa_filtered[["uid", "question", "answer", "source_files_parsed", "difficulty"]]

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

SELECTED_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
YEAR_SET = set(SELECTED_YEARS)

allow_patterns = [
    f"treasury_bulletins_parsed/transformed/treasury_bulletin_{year}_*.txt"
    for year in SELECTED_YEARS
]

local_dir = snapshot_download(
    repo_id="databricks/officeqa",
    repo_type="dataset",
    allow_patterns=allow_patterns,
    local_dir="officeqa_hf_data"
)

txt_files = sorted(Path(local_dir).rglob("treasury_bulletin_*.txt"))

print("Downloaded text files:", len(txt_files))
print("First 10 files:")
for file in txt_files[:10]:
    print(file.name)

In [ ]:
required_source_files = sorted({
    file
    for files in qa_filtered["source_files_parsed"]
    for file in files
})

downloaded_file_names = sorted([path.name for path in txt_files])

missing_files = sorted(set(required_source_files) - set(downloaded_file_names))

print("Required source files from answer key:")
for file in required_source_files:
    print(file)

print("\nMissing files:")
print(missing_files)

assert len(missing_files) == 0, "Some required files are missing. Check download step."

In [ ]:
def read_documents(txt_files):
    docs = []

    for path in txt_files:
        text = path.read_text(encoding="utf-8", errors="ignore")

        docs.append({
            "source_file": path.name,
            "year": get_year(path.name),
            "month": get_month(path.name),
            "text": text
        })

    return docs

docs = read_documents(txt_files)

print("Documents loaded:", len(docs))
print("Example document:")
print(docs[0]["source_file"], docs[0]["year"], docs[0]["month"])

In [ ]:
def simple_chunks(text, chunk_size=1000):
    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]

        if chunk.strip():
            chunks.append(chunk)

    return chunks


baseline_chunks = []

for doc in docs:
    chunks = simple_chunks(doc["text"], chunk_size=1000)

    for i, chunk in enumerate(chunks):
        baseline_chunks.append({
            "id": f"baseline_{doc['source_file']}_{i}",
            "text": chunk,
            "metadata": {
                "source_file": doc["source_file"],
                "year": int(doc["year"]),
                "month": int(doc["month"]),
                "chunk_id": int(i),
                "system": "baseline"
            }
        })

print("Baseline chunks created:", len(baseline_chunks))
print("Example baseline metadata:")
print(baseline_chunks[0]["metadata"])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1600,
    chunk_overlap=250,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""]
)

engineered_chunks = []

for doc in docs:
    chunks = splitter.split_text(doc["text"])

    for i, chunk in enumerate(chunks):
        engineered_chunks.append({
            "id": f"engineered_{doc['source_file']}_{i}",
            "text": chunk,
            "metadata": {
                "source_file": doc["source_file"],
                "year": int(doc["year"]),
                "month": int(doc["month"]),
                "chunk_id": int(i),
                "system": "engineered"
            }
        })

print("Engineered chunks created:", len(engineered_chunks))
print("Example engineered metadata:")
print(engineered_chunks[0]["metadata"])

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="./chroma_db_financial_rag")

def embed_texts(texts):
    return embedder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False
    ).tolist()

def create_collection(collection_name, chunks):
    try:
        client.delete_collection(collection_name)
    except:
        pass

    collection = client.create_collection(name=collection_name)

    batch_size = 100

    for i in tqdm(range(0, len(chunks), batch_size)):
        batch = chunks[i:i + batch_size]
        texts = [item["text"] for item in batch]

        collection.add(
            ids=[item["id"] for item in batch],
            documents=texts,
            metadatas=[item["metadata"] for item in batch],
            embeddings=embed_texts(texts)
        )

    return collection

baseline_collection = create_collection("baseline_rag_v1", baseline_chunks)
engineered_collection = create_collection("engineered_rag_v1", engineered_chunks)

print("Baseline collection count:", baseline_collection.count())
print("Engineered collection count:", engineered_collection.count())

In [ ]:
sample_question = qa_filtered.loc[0, "question"]

print("Sample question:")
print(sample_question)

query_embedding = embed_texts([sample_question])[0]

results = baseline_collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

print("\nTop 5 retrieved files:")
for meta, distance in zip(results["metadatas"][0], results["distances"][0]):
    print(meta["source_file"], "distance:", distance)

In [ ]:
def evaluate_retriever(qa_df, retrieve_function, system_name, k=5):
    results = []

    for _, row in qa_df.iterrows():
        question = row["question"]
        gold_files = set(row["source_files_parsed"])

        retrieved = retrieve_function(question, k=k)
        retrieved_files = [r["metadata"]["source_file"] for r in retrieved]

        first_correct_rank = None

        for rank, file in enumerate(retrieved_files, start=1):
            if file in gold_files:
                first_correct_rank = rank
                break

        hit = 1 if first_correct_rank else 0
        mrr = 1 / first_correct_rank if first_correct_rank else 0
        recall = len(set(retrieved_files) & gold_files) / len(gold_files)

        results.append({
            "system": system_name,
            "uid": row["uid"],
            "question": question,
            "gold_files": list(gold_files),
            "retrieved_files": retrieved_files,
            "hit_at_5": hit,
            "mrr": mrr,
            "recall_at_5": recall
        })

    results_df = pd.DataFrame(results)

    summary = {
        "system": system_name,
        "hit_rate_at_5": results_df["hit_at_5"].mean(),
        "mrr": results_df["mrr"].mean(),
        "recall_at_5": results_df["recall_at_5"].mean()
    }

    return results_df, summary

In [ ]:
def retrieve_baseline(question, k=5):
    query_embedding = embed_texts([question])[0]

    results = baseline_collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    output = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        output.append({
            "text": doc,
            "metadata": meta,
            "distance": distance
        })

    return output

In [ ]:
MONTHS = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}

def metadata_filter_from_question(question):
    q = question.lower()

    years = [
        int(y) for y in re.findall(r"\b(20\d{2}|19\d{2})\b", q)
        if int(y) in YEAR_SET
    ]

    months = [
        num for name, num in MONTHS.items()
        if name in q
    ]

    if len(years) == 1 and len(months) == 1:
        return {"$and": [{"year": years[0]}, {"month": months[0]}]}
    elif len(years) == 1:
        return {"year": years[0]}
    elif len(months) == 1:
        return {"month": months[0]}
    else:
        return None


def retrieve_engineered(question, k=5):
    query_embedding = embed_texts([question])[0]
    where_filter = metadata_filter_from_question(question)

    if where_filter:
        results = engineered_collection.query(
            query_embeddings=[query_embedding],
            n_results=k,
            where=where_filter,
            include=["documents", "metadatas", "distances"]
        )
    else:
        results = engineered_collection.query(
            query_embeddings=[query_embedding],
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )

    output = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        output.append({
            "text": doc,
            "metadata": meta,
            "distance": distance
        })

    return output

In [ ]:
baseline_retriever_details, baseline_retriever_summary = evaluate_retriever(
    qa_filtered,
    retrieve_baseline,
    "Baseline",
    k=5
)

engineered_retriever_details, engineered_retriever_summary = evaluate_retriever(
    qa_filtered,
    retrieve_engineered,
    "Engineered",
    k=5
)

print("Baseline Retriever Summary:")
print(baseline_retriever_summary)

print("\nEngineered Retriever Summary:")
print(engineered_retriever_summary)

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()

In [ ]:
manual_eval_rows = []

for _, row in qa_filtered.iterrows():
    question = row["question"]
    gold_answer = row["answer"]

    baseline_retrieved = retrieve_baseline(question, k=5)
    engineered_retrieved = retrieve_engineered(question, k=5)

    manual_eval_rows.append({
        "uid": row["uid"],
        "question": question,
        "gold_answer": gold_answer,
        "gold_files": row["source_files_parsed"],
        "baseline_retrieved_files": [r["metadata"]["source_file"] for r in baseline_retrieved],
        "engineered_retrieved_files": [r["metadata"]["source_file"] for r in engineered_retrieved],
        "baseline_grounded": "",
        "baseline_factual_correct": "",
        "baseline_hallucination": "",
        "engineered_grounded": "",
        "engineered_factual_correct": "",
        "engineered_hallucination": ""
    })

manual_eval_df = pd.DataFrame(manual_eval_rows)

manual_eval_df

In [ ]:
manual_eval_df.to_csv("manual_generator_evaluation_template.csv", index=False)

In [ ]:
def inspect_question(row_number, system="engineered", max_chars=2500):
    row = qa_filtered.loc[row_number]

    question = row["question"]
    gold_answer = row["answer"]
    gold_files = row["source_files_parsed"]

    if system.lower() == "baseline":
        retrieved = retrieve_baseline(question, k=5)
    else:
        retrieved = retrieve_engineered(question, k=5)

    print("=" * 100)
    print("ROW:", row_number)
    print("UID:", row["uid"])
    print("SYSTEM:", system.upper())
    print("\nQUESTION:")
    print(question)

    print("\nGOLD ANSWER FROM OFFICEQA:")
    print(gold_answer)

    print("\nGOLD SOURCE FILE(S):")
    print(gold_files)

    print("\nRETRIEVED FILES:")
    for i, item in enumerate(retrieved, start=1):
        meta = item["metadata"]
        print(f"{i}. {meta['source_file']} | year={meta['year']} | month={meta['month']} | chunk={meta['chunk_id']}")

    print("\nRETRIEVED CHUNK TEXT:")
    for i, item in enumerate(retrieved, start=1):
        meta = item["metadata"]
        print("\n" + "-" * 100)
        print(f"TOP {i}: {meta['source_file']} | chunk={meta['chunk_id']}")
        print("-" * 100)
        print(item["text"][:max_chars])

In [ ]:
inspect_question(0, system="engineered")

In [ ]:
def metadata_filter_from_question_v2(question):
    """
    Improved metadata filter:
    Use year filtering only.

    Reason:
    The month mentioned in the question may refer to the report date,
    not the Treasury Bulletin publication month.
    Example: March 31, 2025 data may appear in treasury_bulletin_2025_06.txt.
    """
    q = question.lower()

    years = [
        int(y) for y in re.findall(r"\b(20\d{2}|19\d{2})\b", q)
        if int(y) in YEAR_SET
    ]

    if len(years) == 1:
        return {"year": years[0]}
    else:
        return None


def retrieve_engineered(question, k=5):
    query_embedding = embed_texts([question])[0]
    where_filter = metadata_filter_from_question_v2(question)

    if where_filter:
        results = engineered_collection.query(
            query_embeddings=[query_embedding],
            n_results=k,
            where=where_filter,
            include=["documents", "metadatas", "distances"]
        )
    else:
        results = engineered_collection.query(
            query_embeddings=[query_embedding],
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )

    output = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        output.append({
            "text": doc,
            "metadata": meta,
            "distance": distance
        })

    return output

In [ ]:
engineered_retriever_details, engineered_retriever_summary = evaluate_retriever(
    qa_filtered,
    retrieve_engineered,
    "Engineered",
    k=5
)

print("Updated Engineered Retriever Summary:")
print(engineered_retriever_summary)

In [ ]:
inspect_question(0, system="engineered")

In [ ]:
baseline_retriever_details, baseline_retriever_summary = evaluate_retriever(
    qa_filtered,
    retrieve_baseline,
    "Baseline",
    k=5
)

engineered_retriever_details, engineered_retriever_summary = evaluate_retriever(
    qa_filtered,
    retrieve_engineered,
    "Engineered",
    k=5
)

print("Baseline Retriever Summary:")
print(baseline_retriever_summary)

print("\nUpdated Engineered Retriever Summary:")
print(engineered_retriever_summary)

In [ ]:
for i in range(len(qa_filtered)):
    inspect_question(i, system="engineered", max_chars=2000)

In [ ]:
for i in range(len(qa_filtered)):
    inspect_question(i, system="baseline", max_chars=2000)

In [ ]:
manual_scores = {
    "UID0010": {
        "baseline_grounded": 0,
        "baseline_factual_correct": 0,
        "baseline_hallucination": 1,
        "engineered_grounded": 0,
        "engineered_factual_correct": 0,
        "engineered_hallucination": 1
    },
    "UID0042": {
        "baseline_grounded": 0,
        "baseline_factual_correct": 0,
        "baseline_hallucination": 1,
        "engineered_grounded": 0,
        "engineered_factual_correct": 0,
        "engineered_hallucination": 1
    },
    "UID0054": {
        "baseline_grounded": 0,
        "baseline_factual_correct": 0,
        "baseline_hallucination": 1,
        "engineered_grounded": 0,
        "engineered_factual_correct": 0,
        "engineered_hallucination": 1
    },
    "UID0066": {
        "baseline_grounded": 0,
        "baseline_factual_correct": 0,
        "baseline_hallucination": 1,
        "engineered_grounded": 0,
        "engineered_factual_correct": 0,
        "engineered_hallucination": 1
    },
    "UID0081": {
        "baseline_grounded": 0,
        "baseline_factual_correct": 0,
        "baseline_hallucination": 1,
        "engineered_grounded": 0,
        "engineered_factual_correct": 0,
        "engineered_hallucination": 1
    },
    "UID0086": {
        "baseline_grounded": 0,
        "baseline_factual_correct": 0,
        "baseline_hallucination": 1,
        "engineered_grounded": 1,
        "engineered_factual_correct": 1,
        "engineered_hallucination": 0
    },
    "UID0098": {
        "baseline_grounded": 0,
        "baseline_factual_correct": 0,
        "baseline_hallucination": 1,
        "engineered_grounded": 0,
        "engineered_factual_correct": 0,
        "engineered_hallucination": 1
    },
    "UID0099": {
        "baseline_grounded": 0,
        "baseline_factual_correct": 0,
        "baseline_hallucination": 1,
        "engineered_grounded": 0,
        "engineered_factual_correct": 0,
        "engineered_hallucination": 1
    },
    "UID0102": {
        "baseline_grounded": 0,
        "baseline_factual_correct": 0,
        "baseline_hallucination": 1,
        "engineered_grounded": 1,
        "engineered_factual_correct": 1,
        "engineered_hallucination": 0
    }
}

In [ ]:
def metadata_filter_from_question_v3(question):
    """
    Improved metadata filter:
    - Deduplicates years.
    - Uses year-only filtering.
    - For year-end / CY questions, also includes the following year,
      because year-end data often appears in the next Treasury Bulletin.
    """
    q = question.lower()

    years = sorted(set([
        int(y) for y in re.findall(r"\b(20\d{2}|19\d{2})\b", q)
        if int(y) in YEAR_SET
    ]))

    if len(years) == 0:
        return None

    expanded_years = set(years)

    if "year-end" in q or "cy" in q or "calendar year" in q:
        for y in years:
            if y + 1 in YEAR_SET:
                expanded_years.add(y + 1)

    expanded_years = sorted(expanded_years)

    if len(expanded_years) == 1:
        return {"year": expanded_years[0]}
    else:
        return {"year": {"$in": expanded_years}}

In [ ]:
def retrieve_engineered(question, k=5):
    query_embedding = embed_texts([question])[0]
    where_filter = metadata_filter_from_question_v3(question)

    if where_filter:
        results = engineered_collection.query(
            query_embeddings=[query_embedding],
            n_results=k,
            where=where_filter,
            include=["documents", "metadatas", "distances"]
        )
    else:
        results = engineered_collection.query(
            query_embeddings=[query_embedding],
            n_results=k,
            include=["documents", "metadatas", "distances"]
        )

    output = []

    for doc, meta, distance in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        output.append({
            "text": doc,
            "metadata": meta,
            "distance": distance
        })

    return output

In [ ]:
inspect_question(5, system="engineered", max_chars=2000)

In [ ]:
inspect_question(5, system="engineered", max_chars=6000)

In [ ]:
row = qa_filtered.loc[5]
question = row["question"]

retrieved = retrieve_engineered(question, k=5)

for i, item in enumerate(retrieved, start=1):
    text = item["text"]
    meta = item["metadata"]

    print("=" * 100)
    print(f"TOP {i}: {meta['source_file']} | chunk={meta['chunk_id']}")

    for keyword in ["Total assets", "Total Assets", "Assets", "June 30, 2022", "September 30, 2022"]:
        if keyword in text:
            print(f"Found keyword: {keyword}")

    lines = text.splitlines()
    for line in lines:
        if "Total assets" in line or "Total Assets" in line or "Total liabilities" in line:
            print(line)

In [ ]:
source_file_to_check = "treasury_bulletin_2022_12.txt"
chunk_ids_to_check = [252, 253, 254, 255]

for target_chunk_id in chunk_ids_to_check:
    print("=" * 120)
    print(f"{source_file_to_check} | chunk {target_chunk_id}")
    print("=" * 120)

    found = False

    for item in engineered_chunks:
        meta = item["metadata"]

        if meta["source_file"] == source_file_to_check and meta["chunk_id"] == target_chunk_id:
            print(item["text"][:6000])
            found = True
            break

    if not found:
        print("Chunk not found.")

In [ ]:
def get_neighbor_chunks(retrieved_chunks, all_chunks, neighbor_window=1):
    """
    Add nearby chunks from the same source file.
    This helps when a table heading is retrieved but the table rows continue
    in the next chunk.
    """
    expanded = []
    seen_ids = set()

    chunk_lookup = {}

    for item in all_chunks:
        meta = item["metadata"]
        key = (meta["source_file"], meta["chunk_id"])
        chunk_lookup[key] = item

    for retrieved in retrieved_chunks:
        meta = retrieved["metadata"]
        source_file = meta["source_file"]
        chunk_id = meta["chunk_id"]

        for offset in range(-neighbor_window, neighbor_window + 1):
            neighbor_id = chunk_id + offset
            key = (source_file, neighbor_id)

            if key in chunk_lookup:
                neighbor = chunk_lookup[key]
                unique_id = (neighbor["metadata"]["source_file"], neighbor["metadata"]["chunk_id"])

                if unique_id not in seen_ids:
                    expanded.append({
                        "text": neighbor["text"],
                        "metadata": neighbor["metadata"],
                        "distance": retrieved.get("distance", None)
                    })
                    seen_ids.add(unique_id)

    return expanded

In [ ]:
def retrieve_engineered_for_generation(question, k=5):
    top_chunks = retrieve_engineered(question, k=k)

    expanded_chunks = get_neighbor_chunks(
        retrieved_chunks=top_chunks,
        all_chunks=engineered_chunks,
        neighbor_window=1
    )

    return expanded_chunks

In [ ]:
row = qa_filtered.loc[5]
question = row["question"]

expanded_retrieved = retrieve_engineered_for_generation(question, k=5)

for i, item in enumerate(expanded_retrieved, start=1):
    text = item["text"]
    meta = item["metadata"]

    print("=" * 120)
    print(f"EXPANDED {i}: {meta['source_file']} | chunk={meta['chunk_id']}")
    print("=" * 120)

    if "Total assets" in text or "Total Assets" in text:
        print("FOUND TOTAL ASSETS LINE")

    lines = text.splitlines()
    for line in lines:
        if "Total assets" in line or "Total Assets" in line or "Total liabilities" in line:
            print(line)

In [ ]:
row = qa_filtered.loc[8]
question = row["question"]

expanded_retrieved = retrieve_engineered_for_generation(question, k=5)

for i, item in enumerate(expanded_retrieved, start=1):
    text = item["text"]
    meta = item["metadata"]

    print("=" * 120)
    print(f"EXPANDED {i}: {meta['source_file']} | chunk={meta['chunk_id']}")
    print("=" * 120)

    lines = text.splitlines()

    for line in lines:
        if (
            "Corporate income taxes" in line
            or "Corporation income taxes" in line
            or "Corporate" in line
            or "Corporation" in line
            or "First-Quarter Net Budget Receipts" in line
            or "Second-Quarter Net Budget Receipts" in line
            or "Third-Quarter Net Budget Receipts" in line
            or "Fourth-Quarter Net Budget Receipts" in line
        ):
            print(line)

In [ ]:
target_files = [
    "treasury_bulletin_2021_03.txt",
    "treasury_bulletin_2021_06.txt",
    "treasury_bulletin_2021_09.txt",
    "treasury_bulletin_2021_12.txt"
]

for target_file in target_files:
    print("=" * 120)
    print(target_file)
    print("=" * 120)

    for item in engineered_chunks:
        meta = item["metadata"]
        text = item["text"]

        if meta["source_file"] == target_file:
            if (
                "Corporate income taxes" in text
                or "Corporation income taxes" in text
                or "First-Quarter Net Budget Receipts" in text
                or "Second-Quarter Net Budget Receipts" in text
                or "Third-Quarter Net Budget Receipts" in text
                or "Fourth-Quarter Net Budget Receipts" in text
            ):
                print("\n" + "-" * 100)
                print(f"chunk={meta['chunk_id']}")
                print("-" * 100)

                lines = text.splitlines()
                for line in lines:
                    if (
                        "Corporate income taxes" in line
                        or "Corporation income taxes" in line
                        or "First-Quarter Net Budget Receipts" in line
                        or "Second-Quarter Net Budget Receipts" in line
                        or "Third-Quarter Net Budget Receipts" in line
                        or "Fourth-Quarter Net Budget Receipts" in line
                        or "| Source |" in line
                        or "| --- |" in line
                    ):
                        print(line)

In [ ]:
target_file = "treasury_bulletin_2021_12.txt"

for target_chunk_id in range(38, 48):
    print("=" * 120)
    print(f"{target_file} | chunk {target_chunk_id}")
    print("=" * 120)

    for item in engineered_chunks:
        meta = item["metadata"]

        if meta["source_file"] == target_file and meta["chunk_id"] == target_chunk_id:
            print(item["text"][:5000])
            break

In [ ]:
target_file = "treasury_bulletin_2021_12.txt"

raw_text = None

for doc in docs:
    if doc["source_file"] == target_file:
        raw_text = doc["text"]
        break

assert raw_text is not None, "Target file not found."

# Print every location where Corporate income taxes appears
import re

matches = list(re.finditer("Corporate income taxes", raw_text, flags=re.IGNORECASE))

print("Number of matches:", len(matches))

for i, match in enumerate(matches):
    start = max(0, match.start() - 1200)
    end = min(len(raw_text), match.end() + 1200)

    print("=" * 120)
    print(f"MATCH {i+1}")
    print("=" * 120)
    print(raw_text[start:end])

In [ ]:
keywords = [
    "Fourth-Quarter Net Budget Receipts",
    "July",
    "August",
    "September",
    "Corporate income taxes",
    "Corporation income taxes"
]

for keyword in keywords:
    print("=" * 120)
    print("KEYWORD:", keyword)
    print("=" * 120)

    matches = list(re.finditer(keyword, raw_text, flags=re.IGNORECASE))
    print("Matches:", len(matches))

    for match in matches[:10]:
        start = max(0, match.start() - 800)
        end = min(len(raw_text), match.end() + 800)
        print(raw_text[start:end])
        print("-" * 120)

In [ ]:
import numpy as np
from decimal import Decimal, ROUND_HALF_UP

def round_half_up(value, digits=1):
    q = Decimal("1." + "0" * digits)
    return float(Decimal(str(value)).quantize(q, rounding=ROUND_HALF_UP))

values = np.array([
    9.2, -3.2, 62.9,
    16.5, 3.8, 15.3,
    72.8, 13.8, 74.2,
    16.9, 3.0, 86.7
])

q1_raw = np.percentile(values, 25, method="linear")
q3_raw = np.percentile(values, 75, method="linear")

q1 = round_half_up(q1_raw, 1)
q3 = round_half_up(q3_raw, 1)

h_spread = round(q3 - q1, 2)

print("Q1 raw:", q1_raw)
print("Q3 raw:", q3_raw)
print("Q1 rounded:", q1)
print("Q3 rounded:", q3)
print("H-spread:", h_spread)

In [ ]:
for idx, row in manual_eval_df.iterrows():
    uid = row["uid"]
    if uid in manual_scores:
        for col, value in manual_scores[uid].items():
            manual_eval_df.loc[idx, col] = value

manual_eval_df

In [ ]:
print("Baseline Generator Summary:")
print(f"Groundedness: {baseline_generator_summary['groundedness'] * 100:.2f}%")
print(f"Factual Accuracy: {baseline_generator_summary['factual_accuracy'] * 100:.2f}%")
print(f"Hallucination Rate: {baseline_generator_summary['hallucination_rate'] * 100:.2f}%")

print("\nEngineered Generator Summary:")
print(f"Groundedness: {engineered_generator_summary['groundedness'] * 100:.2f}%")
print(f"Factual Accuracy: {engineered_generator_summary['factual_accuracy'] * 100:.2f}%")
print(f"Hallucination Rate: {engineered_generator_summary['hallucination_rate'] * 100:.2f}%")

In [ ]:
baseline_retriever_details, baseline_retriever_summary = evaluate_retriever(
    qa_filtered,
    retrieve_baseline,
    "Baseline",
    k=5
)

engineered_retriever_details, engineered_retriever_summary = evaluate_retriever(
    qa_filtered,
    retrieve_engineered,
    "Engineered",
    k=5
)

print("Baseline Retriever Summary:")
print(baseline_retriever_summary)

print("\nEngineered Retriever Summary:")
print(engineered_retriever_summary)

## Final Scorecard

In [ ]:
scorecard = pd.DataFrame({
    "Metric": [
        "Hit Rate (K=5)",
        "MRR",
        "Groundedness",
        "Factual Accuracy",
        "Hallucination Rate"
    ],
    "Baseline (Simple)": [
        f"{baseline_retriever_summary['hit_rate_at_5'] * 100:.2f}%",
        f"{baseline_retriever_summary['mrr']:.2f}",
        f"{baseline_generator_summary['groundedness'] * 100:.2f}%",
        f"{baseline_generator_summary['factual_accuracy'] * 100:.2f}%",
        f"{baseline_generator_summary['hallucination_rate'] * 100:.2f}%"
    ],
    "Engineered (Improved)": [
        f"{engineered_retriever_summary['hit_rate_at_5'] * 100:.2f}%",
        f"{engineered_retriever_summary['mrr']:.2f}",
        f"{engineered_generator_summary['groundedness'] * 100:.2f}%",
        f"{engineered_generator_summary['factual_accuracy'] * 100:.2f}%",
        f"{engineered_generator_summary['hallucination_rate'] * 100:.2f}%"
    ]
})

scorecard

In [ ]:
print(scorecard.to_string(index=False))

In [ ]:
scorecard.to_csv("final_scorecard.csv", index=False)
manual_eval_df.to_csv("manual_generator_evaluation_completed.csv", index=False)
baseline_retriever_details.to_csv("baseline_retriever_details.csv", index=False)
engineered_retriever_details.to_csv("engineered_retriever_details.csv", index=False)

In [ ]:
from google.colab import files

files.download("final_scorecard.csv")
files.download("manual_generator_evaluation_completed.csv")
files.download("baseline_retriever_details.csv")
files.download("engineered_retriever_details.csv")